# M1 — Retrieval-Augmented Generation

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

In [ ]:
# --- Step 1. Install dependencies
!pip -q install datasets sentence-transformers faiss-cpu transformers accelerate nltk bitsandbytes

Step 2. Imports, paths, and basic setup

In [ ]:
import os, json, math, re, faiss, numpy as np, torch
from datasets import load_from_disk, Dataset
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

In [ ]:
# ===== path =====
DS_ROOT = "/Volumes/main/default/thesis_project/QASPER/qasper_datasetdict"  # DatasetDict root
OUT_DIR = "/Volumes/main/default/thesis_project/QASPER"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== loading validation split =====
try:
    ds_val = load_from_disk(DS_ROOT)["test"]
except Exception as e:
    # If error:
    ARROW_PATH = "/Volumes/main/default/thesis_project/QASPER/qasper_datasetdict/test/data-00000-of-00001.arrow"
    ds_val = Dataset.from_file(ARROW_PATH)

print(len(ds_val))
print(ds_val.features)


Step 3. Chunking utilities (title/abstract/full_text -> paragraph chunks)

In [ ]:
def chunk_by_sent(text, max_chars=1200, overlap_chars=200):
    """
    Greedy sentence-based chunking to ~max_chars per chunk.
    Adds simple character overlap between adjacent chunks for recall stability.
    """
    sents = sent_tokenize(text)
    chunks, buf = [], ""
    for s in sents:
        if len(buf) + len(s) + 1 <= max_chars:
            buf = (buf + " " + s).strip()
        else:
            if buf:
                chunks.append(buf)
            buf = s
    if buf:
        chunks.append(buf)

    # Add tail-head overlap to retain context continuity
    if overlap_chars and len(chunks) >= 2:
        merged = []
        for i, c in enumerate(chunks):
            if i == 0:
                merged.append(c)
            else:
                prev_tail = chunks[i-1][-overlap_chars:]
                merged.append((prev_tail + " " + c).strip())
        chunks = merged
    return chunks


In [ ]:
def extract_val_chunks(ds_val, max_chars=1200, overlap_chars=200):
    """
    Build a paragraph/chunk corpus from the validation split.
    Uses only original paper text: title, abstract, full_text.
    Does NOT use Q/A, evidence, or any gold labels.
    Compatible with your schema:
       full_text.section_name: List[str]
       full_text.paragraphs  : List[List[str]]
    """
    corpus = []
    for ex in ds_val:
        pid = ex["id"]

        # Title + abstract
        for k in ("title", "abstract"):
            txt = (ex.get(k) or "").strip()
            if txt:
                for j, ck in enumerate(chunk_by_sent(txt, max_chars, overlap_chars)):
                    corpus.append({"paper_id": pid, "chunk_id": f"{pid}::{k}#c{j}", "text": ck})

        # Full text (section_name + paragraphs)
        ft = ex.get("full_text") or {}
        secs = ft.get("section_name") or []
        plist = ft.get("paragraphs") or []  # List[List[str]]
        n = min(len(secs), len(plist))      # defensive bounds
        for si in range(n):
            sec = (secs[si] or "").strip()
            for pi, p in enumerate(plist[si] or []):
                txt = (f"{sec} {p}".strip() if sec else (p or "").strip())
                if not txt:
                    continue
                for cj, ck in enumerate(chunk_by_sent(txt, max_chars, overlap_chars)):
                    corpus.append(
                        {"paper_id": pid, "chunk_id": f"{pid}::s{si}p{pi}#c{cj}", "text": ck}
                    )
    return corpus

In [ ]:
# Fix NLTK sentence tokenizer resources
import nltk

# Try to locate; if missing, download quietly
for pkg in ["punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)

# Quick sanity check
from nltk.tokenize import sent_tokenize
print(sent_tokenize("This is a test. It should split sentences! Does it? Yes."))

In [ ]:
# Build corpus (validation only)
corpus = extract_val_chunks(ds_val, max_chars=1200, overlap_chars=200)
print("Total chunks (validation):", len(corpus))
print("Sample chunk:", corpus[0])

Step 4. Embeddings on GPU + FAISS index on CPU

In [ ]:
# Enable TF32 on A100 for speed (safe for inference)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
EMB_NAME = "intfloat/e5-large-v2"   # good quality; uses query:/passage: prefixes
emb = SentenceTransformer(EMB_NAME, device="cuda")  # <--- run embeddings on GPU

In [ ]:
def emb_texts(texts, mode="passage", batch=512, workers=4):
    """
    Encode texts on GPU with e5-large-v2.
    - Use prefix 'passage:' for corpus, 'query:' for queries.
    - normalize_embeddings=True allows cosine ≈ inner product on normalized vectors.
    """
    prefix = "passage: " if mode == "passage" else "query: "
    vecs = emb.encode(
        [prefix + t for t in texts],
        batch_size=batch,              # try 512~1024 on A100; reduce if OOM
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
        num_workers=workers,
    )
    return vecs.astype("float32")

In [ ]:
# Compute embeddings for all chunks (GPU), then build FAISS index (CPU)
texts = [r["text"] for r in corpus]
X = emb_texts(texts, mode="passage", batch=512, workers=4)

In [ ]:
index = faiss.IndexFlatIP(X.shape[1])  # with normalized vectors, IP == cosine
index.add(X)

In [ ]:
INDEX_PATH = "/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_e5.index"
META_PATH  = "/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_meta.jsonl"
faiss.write_index(index, INDEX_PATH)
with open(META_PATH, "w") as f:
    for r in corpus:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved index to:", INDEX_PATH)
print("Saved meta  to:", META_PATH)

Step 5. Retrieval (restrict to the same paper_id)

In [ ]:
# Load index/meta back (simulates a fresh session)
index = faiss.read_index(INDEX_PATH)
meta  = [json.loads(l) for l in open(META_PATH, "r")]
NALL = len(meta)

In [ ]:
def embed_query(q):
    # Query embedding on GPU (small batch)
    return emb_texts([q], mode="query", batch=64)

In [ ]:
def retrieve(query, paper_id=None, top_k=6, widen=60, step=60, max_widen=None):
    """
    Retrieve top_k chunks for a query.
    Strategy:
      1) Search with a relatively large candidate pool (widen).
      2) Filter by paper_id if provided (recommended for QASPER).
      3) If filtered results < top_k, increase widen and repeat (up to max_widen).
    """
    if max_widen is None:
        max_widen = min(NALL, 1000)
    qv = embed_query(query)
    K = min(widen, max_widen)
    while True:
        D, I = index.search(qv, K)
        cand = []
        for rank, i in enumerate(I[0].tolist()):
            m = meta[i]
            if (paper_id is None) or (m["paper_id"] == paper_id):
                cand.append({**m, "score": float(D[0][rank])})
            if len(cand) >= top_k:
                break
        if len(cand) >= top_k or K >= max_widen:
            return cand[:top_k]
        K = min(K + step, max_widen)

Step 6. LLM: Mistral-7B-Instruct + RAG prompt (generation capped at 192 tokens)

In [ ]:
!pip -q install -U "transformers>=4.41" "accelerate>=0.31" "bitsandbytes>=0.43" huggingface_hub hf-transfer

In [ ]:
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, time

GEN_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
USE_4BIT = True  # keep True for memory/speed; you can set False to use bf16 weights directly

In [ ]:
if USE_4BIT:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    lm = AutoModelForCausalLM.from_pretrained(
        GEN_NAME, device_map="auto",
        quantization_config=bnb, torch_dtype=torch.bfloat16
    )
else:
    lm = AutoModelForCausalLM.from_pretrained(
        GEN_NAME, device_map="auto", torch_dtype=torch.bfloat16
    )

In [ ]:
tok = AutoTokenizer.from_pretrained(GEN_NAME, use_fast=True)

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

In [ ]:
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id
if getattr(lm.config, "pad_token_id", None) is None:
    lm.config.pad_token_id = tok.pad_token_id

In [ ]:
import re

SYS = (
    "You are a careful research assistant. "
    "Use ONLY the provided context to answer. "
    "Prefer EXTRACTIVE answers: copy exact spans, names, or citation tokens as they appear (e.g., BIBREF19). "
    "If the question asks 'which/who/what methods', output ONLY a short list (comma-separated), no explanations. "
    "If the context is insufficient, say exactly: I don't know based on the provided context. "
    "Do NOT repeat the question, context, or any headers. "
    "Keep the answer under 192 tokens."
)

In [ ]:
def clean_ctx_text(t: str) -> str:
    # Do NOT drop BIBREFxx — some gold answers are exactly these tokens.
    t = re.sub(r"\s+", " ", t).strip()
    return t

In [ ]:
def build_prompt(question, ctxs):
    # Clean context and label [p#]
    ctx_lines = [f"[p{i+1}] {clean_ctx_text(c['text'])}" for i, c in enumerate(ctxs)]
    ctx = "\n\n".join(ctx_lines)
    user = (
        f"Question: {question}\n\n"
        f"Context:\n{ctx}\n\n"
        f"Instructions:\n"
        f"- Cite supporting snippets by their ids like [p1], [p3].\n"
        f'- If unsure, say: "I don\'t know based on the provided context."'
    )
    # Put 'Final answer:' inside the prompt so the model starts answering after it
    return f"<s>[INST] <<SYS>>\n{SYS}\n<</SYS>>\n{user}\n\nFinal answer: [/INST]"

In [ ]:
# A tiny dry-run to make sure the prompt looks right
_fake_ctxs = [{"text": "This is context about model A."},
              {"text": "Another paragraph mentioning [p2] style citations."}]
print(build_prompt("What is model A?", _fake_ctxs)[-600:])  # tail of the prompt

Step 7. Build a question table from validation (no gold used) and run a small batch

In [ ]:
import pandas as pd
from tqdm import tqdm

In [ ]:
# Create a (paper_id, question_id, question) DataFrame from validation split
rows = []
for ex in ds_val:
    pid  = ex["id"]
    qs   = ex["qas"]["question"] or []
    qids = ex["qas"]["question_id"] or []
    for q, qid in zip(qs, qids):
        rows.append({"paper_id": pid, "question_id": qid, "question": q})
df_val_qas = pd.DataFrame(rows)
print("Total questions in validation:", len(df_val_qas))
df_val_qas.head()

In [ ]:
# Step 7. Load your standalone Q/A parquet ---
import pandas as pd

In [ ]:
QA_PARQUET = "/Volumes/main/default/thesis_project/QASPER/processed_20250805_162328/qasper_test_qa_evidence.parquet"
df = pd.read_parquet(QA_PARQUET)

In [ ]:
# Identify the question column robustly
question_col = next((c for c in ["question","Question","question_text","q_text","Q"] if c in df.columns), None)
assert question_col is not None, f"Could not find a question column in: {list(df.columns)}"

In [ ]:
# Ensure we have paper_id; if missing but question_id exists, map from ds_val
if "paper_id" not in df.columns:
    if "question_id" in df.columns:
        # ds_val is already loaded in your earlier steps
        qid_to_pid = {}
        for ex in ds_val:
            pid  = ex["id"]
            qids = ex["qas"]["question_id"] or []
            for qid in qids:
                qid_to_pid[qid] = pid
        df["paper_id"] = df["question_id"].map(qid_to_pid)
    else:
        raise ValueError("Neither 'paper_id' nor 'question_id' present; cannot restrict retrieval per paper.")

In [ ]:
# Basic cleanup
df = df.dropna(subset=[question_col, "paper_id"]).reset_index(drop=True)
df[question_col] = df[question_col].astype(str)
df["paper_id"]   = df["paper_id"].astype(str)

print("Rows after cleaning:", len(df))
df.head(3)

Step 8. Run RAG (M1) on a subset for a quick smoke test; then scale up

In [ ]:
# Step 8. Batched same-paper retrieval + Cross-Encoder reranking + generation + save
from tqdm import trange
import os, re, torch, numpy as np
from sentence_transformers import CrossEncoder

In [ ]:
# Throughput knobs
BATCH_GEN   = 16      # try 16/24/32 on A100
TOP_K       = 5       # final number of context chunks per question
WIDEN       = 500     # FAISS candidate pool before same-paper filter
CAND_RERANK = 30      # candidates (within paper) sent to reranker
MAX_NEW     = 192     # generation cap

In [ ]:
RERANK   = True
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda") if RERANK else None

In [ ]:
def rerank_hits(query: str, hits: list, top_k: int = TOP_K):
    """Rerank within-paper candidates and return top_k."""
    if (not RERANK) or len(hits) <= top_k:
        return hits[:top_k]
    pool = hits[:CAND_RERANK]
    pairs = [[query, h["text"]] for h in pool]
    scores = reranker.predict(pairs, convert_to_numpy=True)
    for h, s in zip(pool, scores):
        h["rerank_score"] = float(s)
    pool.sort(key=lambda x: x.get("rerank_score", x["score"]), reverse=True)
    return pool[:top_k]

In [ ]:
def normalize_ctx_text(t: str) -> str:
    """Keep original tokens (incl. BIBREFxx); only collapse whitespace."""
    return re.sub(r"\s+", " ", str(t)).strip()

In [ ]:
# Tokenizer safety for batched decoding
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id
if getattr(lm.config, "pad_token_id", None) is None:
    lm.config.pad_token_id = tok.pad_token_id

In [ ]:
# Truncation budget (prompt + contexts) to leave room for MAX_NEW tokens
ctx_len = getattr(lm.config, "max_position_embeddings", 8192)
MAX_INPUT_LEN = max(512, int(ctx_len) - MAX_NEW - 32)

In [ ]:
# Speed hint on A100
torch.backends.cuda.matmul.allow_tf32 = True

answers, used_ctxs = [], []

In [ ]:
import re

def keep_after_final_answer(text: str) -> str:
    if "Final answer:" in text:
        text = text.split("Final answer:")[-1]
    text = re.sub(r"(?i)<<sys>>.*?<</sys>>", "", text, flags=re.DOTALL)
    text = re.sub(r"(?i)\b(question|context|instructions)\s*:\s*", "", text)
    return text.strip()

In [ ]:
for start in trange(0, len(df), BATCH_GEN):
    sub = df.iloc[start:start + BATCH_GEN]
    q_texts = sub[question_col].astype(str).tolist()
    pids    = sub["paper_id"].astype(str).tolist()

    # 1) Batched query embeddings (GPU)
    Q = emb.encode(
        ["query: " + t for t in q_texts],
        batch_size=256, show_progress_bar=False,
        normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")

    # 2) One FAISS search for the whole batch
    D, I = index.search(Q, WIDEN)   # [B, WIDEN]

    # 3) Same-paper filtering -> pool -> (optional) rerank -> top_k
    ctxs_list = []
    for b in range(len(sub)):
        pid = pids[b]
        pool = []
        for rank, idx in enumerate(I[b].tolist()):
            m = meta[idx]
            if m["paper_id"] == pid:
                pool.append({**m, "score": float(D[b][rank]), "text": normalize_ctx_text(m["text"])})
                if len(pool) >= max(CAND_RERANK, TOP_K):
                    break
        # fallback: if pool still empty, you can either (a) widen more or (b) use title+abstract
        if not pool:
            # (a) simple fallback: use the first TOP_K from I[b] regardless of pid
            for rank, idx in enumerate(I[b].tolist()[:TOP_K]):
                m = meta[idx]
                pool.append({**m, "score": float(D[b][rank]), "text": normalize_ctx_text(m["text"])})
        best = rerank_hits(q_texts[b], pool, top_k=TOP_K)
        ctxs_list.append(best)

    # 4) Build prompts
    prompts = [build_prompt(q_texts[b], ctxs_list[b]) for b in range(len(sub))]

    # 5) Tokenize (left pad + truncation)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
        tok.pad_token_id = tok.eos_token_id
    if getattr(lm.config, "pad_token_id", None) is None:
        lm.config.pad_token_id = tok.pad_token_id

    ctx_len = getattr(lm.config, "max_position_embeddings", 8192)
    MAX_INPUT_LEN = max(512, int(ctx_len) - MAX_NEW - 32)

    inputs = tok(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LEN
    ).to(lm.device)

    # 6) Batched generation (decode ONLY new tokens)
    out = lm.generate(
        **inputs,
        do_sample=False, temperature=0.0, top_p=1.0,
        max_new_tokens=MAX_NEW,
        no_repeat_ngram_size=6,        # optional: reduce instruction echo
        repetition_penalty=1.05,       # optional: slight penalty to repeats
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
        return_dict_in_generate=True
    )

    input_lens = inputs["attention_mask"].sum(dim=1)
    for b in range(len(sub)):
        gen_ids = out.sequences[b, input_lens[b]:]
        ans = tok.decode(gen_ids, skip_special_tokens=True).strip()
        ans = keep_after_final_answer(ans)
        answers.append(ans)
        used_ctxs.append([c["chunk_id"] for c in ctxs_list[b]])

In [ ]:
# 7) Save to your requested path
OUT_PATH = "/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_M1_answers_sample_2.2.parquet"
df_out = df.copy()
df_out["answer_M1"] = answers
df_out["retrieved_chunks"] = used_ctxs
df_out.to_parquet(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Check

In [ ]:
# --- Quick load & peek ---
import pandas as pd

PARQUET = "/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_M1_answers_sample_2.2.parquet"
df = pd.read_parquet(PARQUET)

print(df.shape)              # (rows, cols)
print(df.columns.tolist())   # column names
df.head(3)                   # first few rows


In [ ]:
# --- Show a random sample with key columns (auto-detect question column) ---
cand_cols = ["question", "Question", "question_text", "q_text"]
question_col = next((c for c in cand_cols if c in df.columns), None)

cols = [c for c in ["paper_id", "question_id", question_col, "answer_M1", "retrieved_chunks"] if c in df.columns]
df.sample(5, random_state=42)[cols]

In [ ]:
# --- Simple answer length stats (words) ---
df["ans_len_words"] = df["answer_M1"].str.split().str.len()
df["ans_len_words"].describe()

Apply the same process to Government Report dataset.

In [ ]:
# =========================
# Step 0. Install & imports
# =========================
!pip -q install datasets sentence-transformers faiss-cpu transformers accelerate nltk bitsandbytes pandas pyarrow

import os, json, re, glob, faiss, torch
import numpy as np
import pandas as pd
from tqdm import tqdm, trange

# NLTK sentence tokenizer (with a safe fallback)
import nltk
for pkg in ["punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)
from nltk.tokenize import sent_tokenize
def sent_tokenize_safe(text: str):
    try:
        return sent_tokenize(text)
    except Exception:
        return re.split(r'(?<=[.!?])\s+(?=[A-Z0-9(])', text.strip())


In [ ]:
# ==================================
# Step 1. Paths & basic configuration
# ==================================
# Change to your own paths
GOV_ROOT      = "/Volumes/main/default/thesis_project/GovernmentDocument/gov-report"  # dataset root (has crs/, gao/, split_ids/)
QA_PARQUET    = "/Volumes/main/default/thesis_project/GovernmentDocument/gov-report-qs/processed_20250804_234209/qs_test_qa_evidence.parquet"

OUT_DIR       = "/Volumes/main/default/thesis_project/M1/Test_2.2"  # where to save index/meta/results
os.makedirs(OUT_DIR, exist_ok=True)

INDEX_PATH    = os.path.join(OUT_DIR, "gov_test_e5.index")
META_PATH     = os.path.join(OUT_DIR, "gov_test_meta.jsonl")
RESULT_PARQ   = os.path.join(OUT_DIR, "gov_test_M1_answers_2.2.parquet")


In [ ]:
# ==============================================
# Step 2. Load the QA parquet and inspect schema
# ==============================================
df = pd.read_parquet(QA_PARQUET)
print("QA parquet shape:", df.shape)
print("Columns:", list(df.columns))
df.head(3)

In [ ]:
# ===================================================
# Step 3. Identify key columns (question + document id)
# ===================================================
# Try to detect the question column name
QUESTION_CANDIDATES = ["question", "Question", "question_text", "q_text", "Q"]
question_col = next((c for c in QUESTION_CANDIDATES if c in df.columns), None)
assert question_col is not None, f"Cannot find a question column in {list(df.columns)}"

# Try to detect the document-id column name (used for same-document retrieval)
DOCID_CANDIDATES = ["report_id", "doc_id", "document_id", "paper_id", "id"]
docid_col = next((c for c in DOCID_CANDIDATES if c in df.columns), None)

# If doc-id not present, we will fallback to split_ids (loaded in Step 4) and later map if needed.
print("Detected columns -> question:", question_col, " doc_id:", docid_col)

In [ ]:
# ===================================================
# Step 4. Collect validation IDs from split_ids folder
# ===================================================
# We attempt to read any file under split_ids/*valid*.ids (case-insensitive)
split_dir = os.path.join(GOV_ROOT, "split_ids")
valid_id_files = glob.glob(os.path.join(split_dir, "*valid*.ids")) + glob.glob(os.path.join(split_dir, "*Valid*.ids"))
valid_ids = set()
for fp in valid_id_files:
    with open(fp, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                valid_ids.add(line)

print(f"Found {len(valid_ids)} ids from split_ids (validation). Example:", list(valid_ids)[:5])

# If no split_ids found and docid_col exists in the QA parquet, we can build from there:
if not valid_ids and docid_col is not None:
    valid_ids = set(df[docid_col].astype(str).unique())
    print(f"Built valid_ids from QA parquet '{docid_col}':", len(valid_ids))

In [ ]:
# ===== Fast Step 5. Parse validation reports -> corpus (parallel, fast I/O) =====
!pip -q install orjson blingfire

import os, json, re, glob, orjson
from blingfire import text_to_sentences
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

# 1) Collect candidate files (CRS + GAO), but keep ONLY validation IDs by filename
json_files_all = glob.glob(os.path.join(GOV_ROOT, "crs", "*.json")) + \
                 glob.glob(os.path.join(GOV_ROOT, "gao", "*.json"))

def file_id(fp: str) -> str:
    # filename is the ID per dataset description
    return os.path.splitext(os.path.basename(fp))[0]

if valid_ids:
    json_files = [fp for fp in json_files_all if file_id(fp) in valid_ids]
else:
    json_files = json_files_all  # fallback: keep all

print(f"Total JSON files (all): {len(json_files_all)} | kept for validation: {len(json_files)}")

# 2) Fast sentence-based chunker using blingfire
def chunk_by_sent_fast(text: str, max_chars=1400, overlap_chars=160):
    """
    Faster sentence tokenizer using blingfire.
    Produces ~200–400 token chunks (adjust max_chars/overlap as needed).
    """
    if not text:
        return []
    sents = text_to_sentences(text).split("\n")
    chunks, buf = [], ""
    for s in sents:
        if not s:
            continue
        if len(buf) + len(s) + 1 <= max_chars:
            buf = (buf + " " + s).strip()
        else:
            if buf: chunks.append(buf)
            buf = s
    if buf:
        chunks.append(buf)

    if overlap_chars and len(chunks) >= 2:
        merged = []
        for i, c in enumerate(chunks):
            if i == 0:
                merged.append(c)
            else:
                merged.append((chunks[i-1][-overlap_chars:] + " " + c).strip())
        chunks = merged
    return chunks

# 3) Targeted, light-weight text extractor (avoid deep generic recursion)
KEEP_KEYS = {"title", "summary", "highlight", "report"}

def flatten_report(node, acc: list, depth=0, max_depth=6):
    """
    Collect strings from typical fields; limit recursion depth for speed.
    Only extract str values and common text-like keys.
    """
    if node is None or depth > max_depth:
        return
    if isinstance(node, str):
        t = node.strip()
        if t:
            acc.append(t)
        return
    if isinstance(node, list):
        for x in node:
            flatten_report(x, acc, depth+1, max_depth)
        return
    if isinstance(node, dict):
        # Prefer likely text keys first
        for k in ["text", "paragraph", "paragraphs", "content", "value", "body"]:
            if k in node:
                flatten_report(node[k], acc, depth+1, max_depth)
        # Then traverse children shallowly
        for v in node.values():
            if isinstance(v, (str, list, dict)):
                flatten_report(v, acc, depth+1, max_depth)
        return

def parse_and_chunk_file(fp: str, max_chars=1400, overlap_chars=160):
    """
    Worker: parse one JSON file, extract text, make chunks (title/summary/highlight/report).
    Returns a list of corpus rows for this file.
    """
    try:
        with open(fp, "rb") as f:
            obj = orjson.loads(f.read())
    except Exception as e:
        # Skip corrupt files gracefully
        return []

    rid = str(obj.get("id") or file_id(fp))
    texts = []

    # Title
    title = (obj.get("title") or "").strip()
    if title:
        texts.append(title)

    # CRS: summary; GAO: highlight
    if "summary" in obj:
        flatten_report(obj["summary"], texts, depth=0)
    if "highlight" in obj:
        flatten_report(obj["highlight"], texts, depth=0)

    # Full report (nested)
    if "report" in obj:
        flatten_report(obj["report"], texts, depth=0)

    rows = []
    # chunk each paragraph-ish piece
    for i, p in enumerate(texts):
        if not isinstance(p, str) or not p.strip():
            continue
        for j, ck in enumerate(chunk_by_sent_fast(p, max_chars=max_chars, overlap_chars=overlap_chars)):
            rows.append({"paper_id": rid, "chunk_id": f"{rid}::p{i}#c{j}", "text": ck})
    return rows

# 4) Parallel parse + chunk
MAX_WORKERS = max(1, os.cpu_count() - 1)
worker = partial(parse_and_chunk_file, max_chars=1400, overlap_chars=160)

corpus = []
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(worker, fp) for fp in json_files]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Parsing+chunking (parallel)"):
        rows = fut.result()
        if rows:
            corpus.extend(rows)

print(f"Validation reports processed: {len(json_files)}, total chunks: {len(corpus)}")
if corpus:
    print("Sample chunk:", corpus[0])


In [ ]:
# --- Step 6 (OOM-safe): build embeddings & FAISS index with small batches and sharding ---
import os, gc, faiss, numpy as np, torch

# 1) Free GPU before embedding (unload LLM/reranker if they exist)
for var in ["lm", "reranker"]:
    if var in globals():
        try:
            del globals()[var]
        except Exception:
            pass
gc.collect()
torch.cuda.empty_cache()

# 2) Reduce CUDA fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3) Load encoder on GPU and use shorter max_seq_length + smaller batch
from sentence_transformers import SentenceTransformer

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

EMB_NAME = "intfloat/e5-large-v2"  # (可换成 e5-base-v2 进一步省显存)
emb = SentenceTransformer(EMB_NAME, device="cuda")
emb.max_seq_length = 384  # ↓显存占用；一般对检索影响不大

BATCH_EMB = 128           # 比 512 小很多，更稳
NUM_WORKERS = 2           # DataLoader 线程，别太大以免卡 I/O

def encode_gpu(texts, mode="passage"):
    prefix = "passage: " if mode == "passage" else "query: "
    vecs = emb.encode(
        [prefix + t for t in texts],
        batch_size=BATCH_EMB,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
        num_workers=NUM_WORKERS
    )
    return vecs.astype("float32")

# 4) Streamed/sharded indexing: 不把所有嵌入一次性放内存，边算边 add 到 FAISS
texts = [r["text"] for r in corpus]

def iter_shards(lst, shard_size=5000):
    for i in range(0, len(lst), shard_size):
        yield i, lst[i:i+shard_size]

index = None
DIM = None

for start_idx, shard in iter_shards(texts, shard_size=5000):  # shard_size 可再小一点如 2000
    V = encode_gpu(shard, mode="passage")
    if index is None:
        DIM = V.shape[1]
        index = faiss.IndexFlatIP(DIM)  # 归一化后 IP≈cosine
    index.add(V)
    # 及时让 Python 回收 CPU 内存
    del V
    gc.collect()

# 5) 保存索引 & meta（meta 就是你当前 corpus 的顺序；与 index.add 顺序一致）
INDEX_PATH = "/Volumes/main/default/thesis_project/M1/Test_2.2/govern_test_e5.index"   # 改成你的实际路径
META_PATH  = "/Volumes/main/default/thesis_project/M1/Test_2.2/govern_test_meta.jsonl"

faiss.write_index(index, INDEX_PATH)
with open(META_PATH, "w") as f:
    for r in corpus:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved index:", INDEX_PATH, "| dim =", DIM, "| #chunks =", index.ntotal)

# 6) 如果后面要继续在同一会话里做生成，重新加载 LLM（Step 7）即可

In [ ]:
# --- Step 7. Reload LLM (Mistral-7B-Instruct v0.3) and define the prompt builder ---

import torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1) Speed-friendly matmul on A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# 2) Load the instruction-tuned generator
GEN_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
USE_4BIT = True  # set to False if you prefer bf16 and have enough VRAM

if USE_4BIT:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    lm = AutoModelForCausalLM.from_pretrained(
        GEN_NAME, device_map="auto",
        quantization_config=bnb, torch_dtype=torch.bfloat16
    )
else:
    lm = AutoModelForCausalLM.from_pretrained(
        GEN_NAME, device_map="auto", torch_dtype=torch.bfloat16
    )

tok = AutoTokenizer.from_pretrained(GEN_NAME, use_fast=True)

# 3) Tokenizer safety for batched decoding (left pad + pad token)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id
if getattr(lm.config, "pad_token_id", None) is None:
    lm.config.pad_token_id = tok.pad_token_id

# 4) System instruction (extractive bias + short list answers + strict fallback)
SYS = (
    "You are a careful research assistant. "
    "Use ONLY the provided context to answer. "
    "Prefer EXTRACTIVE answers: copy exact spans, names, or citation tokens as they appear (e.g., BIBREF19). "
    "If the question asks 'which/who/what', output ONLY a short comma-separated list, no explanations. "
    "If the context is insufficient, say exactly: I don't know based on the provided context. "
    "Do NOT repeat the question, context, or any headers. "
    "Keep the answer under 192 tokens."
)

# 5) Context cleaner: keep tokens (incl. BIBREFxx), just collapse whitespace
def clean_ctx_text(t: str) -> str:
    return re.sub(r"\s+", " ", str(t)).strip()

# 6) Prompt builder (adds [p1..pk] labels and 'Final answer:' anchor)
def build_prompt(question: str, ctxs: list) -> str:
    """
    Parameters
    ----------
    question : str
    ctxs     : list of dicts, each with at least {'text': ..., 'chunk_id': ...}
    """
    ctx_lines = [f"[p{i+1}] {clean_ctx_text(c['text'])}" for i, c in enumerate(ctxs)]
    ctx = "\n\n".join(ctx_lines)
    user = (
        f"Question: {question}\n\n"
        f"Context:\n{ctx}\n\n"
        f"Instructions:\n"
        f"- Cite supporting snippets by their ids like [p1], [p3].\n"
        f"- If unsure, say: \"I don't know based on the provided context.\""
    )
    # Place an explicit 'Final answer:' anchor to reduce prompt echo
    return f"<s>[INST] <<SYS>>\n{SYS}\n<</SYS>>\n{user}\n\nFinal answer: [/INST]"

print("Step 7 ready: LLM + tokenizer loaded, prompt builder defined.")


In [ ]:
# --- Step 8. Load QA parquet and prepare (auto-detect columns) ---

import pandas as pd
import os

# >>> Set QA file here <<<
QA_PARQUET = "/Volumes/main/default/thesis_project/GovernmentDocument/gov-report-qs/processed_20250804_234209/qs_test_qa_evidence.parquet"

df = pd.read_parquet(QA_PARQUET)
print("QA parquet shape:", df.shape)
print("Columns:", list(df.columns))

# Detect question column
QUESTION_CANDIDATES = ["question", "Question", "question_text", "q_text", "Q"]
question_col = next((c for c in QUESTION_CANDIDATES if c in df.columns), None)
assert question_col is not None, f"Could not find question column in {list(df.columns)}"

# Detect document id column
DOC_CANDIDATES = ["paper_id", "report_id", "doc_id", "document_id", "id"]
docid_col = next((c for c in DOC_CANDIDATES if c in df.columns), None)

# it often has question_id but no paper_id -> map from ds_val
if docid_col is None and "question_id" in df.columns:
    assert "ds_val" in globals(), "ds_val is required to map question_id -> paper_id for Government Report Dataset."
    qid_to_pid = {}
    for ex in ds_val:
        pid  = ex["id"]
        qids = ex["qas"]["question_id"] or []
        for qid in qids:
            qid_to_pid[qid] = pid
    df["paper_id"] = df["question_id"].map(qid_to_pid)
    docid_col = "paper_id"

# If Government QA contains 'split', keep validation rows only
if "split" in df.columns:
    mask = df["split"].astype(str).str.contains("valid", case=False, na=False)
    if mask.any(): df = df[mask].copy()

# Basic cleanup
df = df.dropna(subset=[question_col, docid_col]).reset_index(drop=True)
df[question_col] = df[question_col].astype(str)
df[docid_col]    = df[docid_col].astype(str)

print(f"Using columns -> question: '{question_col}', doc_id: '{docid_col}'")
print("Rows after cleanup:", len(df))
df.head(3)


In [ ]:
# --- Step 9. Batched same-document retrieval + (optional) reranking + generation + save ---

from tqdm import trange
import numpy as np, torch, re

In [ ]:
# --- Load the CORRECT index/meta pair for Government (and sanity-check) ---
import faiss, json, numpy as np

INDEX_PATH = "/Volumes/main/default/thesis_project/M1/Test_2.2/govern_test_e5.index"
META_PATH  = "/Volumes/main/default/thesis_project/M1/Test_2.2/govern_test_meta.jsonl"

index = faiss.read_index(INDEX_PATH)
with open(META_PATH, "r") as f:
    meta = [json.loads(l) for l in f]

print("Index ntotal:", index.ntotal, "| meta rows:", len(meta))
assert index.ntotal == len(meta), "Index/meta mismatch — rebuild or load the matching pair."

# QA columns for Government
question_col = "question"
docid_col    = "report_id"

# All report_ids in meta (same as 'paper_id' you stored when building corpus)
meta_pids = {m["paper_id"] for m in meta}
missing = set(df[docid_col].astype(str)) - meta_pids
print("Missing report_ids in index:", len(missing))
if missing:
    print("Example missing ids:", list(missing)[:5])  # 如果>0，说明index不是validation或语料不一致

# Quick dry-run to ensure FAISS ids map into meta
Qtest = emb.encode(
    ["query: " + df.iloc[0][question_col]],
    batch_size=1, show_progress_bar=False,
    normalize_embeddings=True, convert_to_numpy=True
).astype("float32")
D_test, I_test = index.search(Qtest, 10)
mx = int(np.max(I_test))
assert 0 <= mx < len(meta), f"FAISS id {mx} out of range for meta size {len(meta)}"
print("Sanity check passed.")

In [ ]:
def emb_query_batch(texts):
    vecs = emb.encode(
        ["query: " + t for t in texts],
        batch_size=256, show_progress_bar=False,
        normalize_embeddings=True, convert_to_numpy=True
    )
    return vecs.astype("float32")

# Optional Cross-Encoder reranker (improves list/which questions)
try:
    from sentence_transformers import CrossEncoder
    RERANK = True
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")
except Exception:
    RERANK = False
    reranker = None
    print("Reranker not available; proceeding without it.")

In [ ]:
def rerank_hits(query: str, hits: list, top_k: int, cand: int = 30):
    if (not RERANK) or len(hits) <= top_k:
        return hits[:top_k]
    pool = hits[:cand]
    pairs = [[query, h["text"]] for h in pool]
    scores = reranker.predict(pairs, convert_to_numpy=True)
    for h, s in zip(pool, scores):
        h["rerank_score"] = float(s)
    pool.sort(key=lambda x: x.get("rerank_score", x["score"]), reverse=True)
    return pool[:top_k]

In [ ]:
def normalize_ctx_text(t: str) -> str:
    # Keep tokens like BIBREFxx; just collapse whitespace
    return re.sub(r"\s+", " ", str(t)).strip()

# Throughput / quality knobs
BATCH_GEN   = 24         # 16 -> 24/32 if VRAM allows
TOP_K       = 4          # 5 -> 4 makes prompts shorter
WIDEN       = 300        # 500 -> 300 is often enough
MAX_NEW     = 192
USE_RERANK  = True       # set False to turn off reranker entirely
CAND_RERANK = 24         # 30 -> 24 a bit faster

# Tokenizer safety for batched left padding + truncation
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id
if getattr(lm.config, "pad_token_id", None) is None:
    lm.config.pad_token_id = tok.pad_token_id

ctx_len = getattr(lm.config, "max_position_embeddings", 8192)
MAX_INPUT_LEN = max(512, int(ctx_len) - MAX_NEW - 32)
torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
# Safety cleaner: if any prompt leaked, keep only text after 'Final answer:'
def keep_after_final_answer(text: str) -> str:
    if "Final answer:" in text:
        text = text.split("Final answer:")[-1]
    text = re.sub(r"(?i)<<sys>>.*?<</sys>>", "", text, flags=re.DOTALL)
    text = re.sub(r"(?i)\b(question|context|instructions)\s*:\s*", "", text)
    return text.strip()

answers, used_ctxs = [], []

In [ ]:
from tqdm import trange
import numpy as np, torch, re

# ==== knobs（先从这里调）====
BATCH_GEN   = 24         # 16 -> 24/32 if VRAM allows
TOP_K       = 4          # 5 -> 4 makes prompts shorter
WIDEN       = 300        # 500 -> 300 is often enough
MAX_NEW     = 192
USE_RERANK  = True       # set False to turn off reranker entirely
CAND_RERANK = 24         # 30 -> 24 a bit faster

# ==== tokenizer & truncation（保持不变）====
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token; tok.pad_token_id = tok.eos_token_id
if getattr(lm.config, "pad_token_id", None) is None:
    lm.config.pad_token_id = tok.pad_token_id
ctx_len = getattr(lm.config, "max_position_embeddings", 8192)
MAX_INPUT_LEN = max(512, int(ctx_len) - MAX_NEW - 32)
torch.backends.cuda.matmul.allow_tf32 = True

def normalize_ctx_text(t: str) -> str:
    return re.sub(r"\s+", " ", str(t)).strip()

# ---- batched reranker (one call per batch) ----
def batch_rerank(q_texts, pools, top_k=TOP_K, cand=CAND_RERANK):
    """
    pools: list[list[hit]] for each query in the batch.
    Returns: list[list[hit]] top_k per query after rerank.
    """
    try:
        from sentence_transformers import CrossEncoder
    except Exception:
        # If ST not available, skip rerank
        return [p[:top_k] for p in pools]

    if not USE_RERANK:
        return [p[:top_k] for p in pools]

    # lazy init once
    global _reranker
    if "_reranker" not in globals() or _reranker is None:
        _reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")

    # build a single big list of pairs
    pairs = []
    sizes = []
    clipped_pools = []
    for qi, pool in enumerate(pools):
        cp = pool[:cand]
        clipped_pools.append(cp)
        sizes.append(len(cp))
        pairs.extend([[q_texts[qi], h["text"]] for h in cp])

    if not pairs:
        return [p[:top_k] for p in pools]

    # one call for the whole batch (larger batch_size speeds it up)
    scores = _reranker.predict(pairs, convert_to_numpy=True, batch_size=512, show_progress_bar=False)

    # scatter back per query and take top_k
    out = []
    off = 0
    for qi, cp in enumerate(clipped_pools):
        sz = sizes[qi]
        sc = scores[off:off+sz]
        off += sz
        for h, s in zip(cp, sc):
            h["rerank_score"] = float(s)
        cp.sort(key=lambda x: x.get("rerank_score", x["score"]), reverse=True)
        out.append(cp[:top_k])
    return out

answers, used_ctxs = [], []

for start in trange(0, len(df), BATCH_GEN):
    sub = df.iloc[start:start + BATCH_GEN]
    q_texts = sub[question_col].astype(str).tolist()
    doc_ids = sub[docid_col].astype(str).tolist()

    # 1) Batched query embeddings
    Q = emb.encode(
        ["query: " + t for t in q_texts],
        batch_size=256, show_progress_bar=False,
        normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")

    # 2) One FAISS search per batch
    D, I = index.search(Q, WIDEN)   # [B, WIDEN]

    # 3) Same-document filtering -> pools (no rerank yet)
    pools = []
    for b in range(len(sub)):
        pid = doc_ids[b]
        pool = []
        rowI = I[b].tolist()
        rowD = D[b].tolist()
        for rank, idx in enumerate(rowI):
            m = meta[idx]
            if m["paper_id"] == pid:
                pool.append({**m, "score": float(rowD[rank]), "text": normalize_ctx_text(m["text"])})
                if len(pool) >= max(CAND_RERANK, TOP_K):
                    break
        if not pool:
            # rare fallback: avoid empty context
            for rank, idx in enumerate(rowI[:TOP_K]):
                m = meta[idx]
                pool.append({**m, "score": float(rowD[rank]), "text": normalize_ctx_text(m["text"])})
        pools.append(pool)

    # 3b) Rerank all queries in ONE call (or keep top_k if disabled)
    ctxs_list = batch_rerank(q_texts, pools, top_k=TOP_K, cand=CAND_RERANK)

    # 4) Build prompts
    prompts = [build_prompt(q_texts[b], ctxs_list[b]) for b in range(len(sub))]

    # 5) Tokenize with explicit truncation
    inputs = tok(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LEN
    ).to(lm.device)

    # 6) Batched generation
    # (Use SDPA kernels; PyTorch usually auto-selects, but we force-enable flash/mem-efficient if available)
    with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=True):
        out = lm.generate(
            **inputs,
            do_sample=False, temperature=0.0, top_p=1.0,
            max_new_tokens=MAX_NEW,
            # no_repeat_ngram_size=6,
            # repetition_penalty=1.05,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.pad_token_id,
            return_dict_in_generate=True
        )

    # 7) Decode ONLY new tokens
    input_lens = inputs["attention_mask"].sum(dim=1)
    for b in range(len(sub)):
        gen_ids = out.sequences[b, input_lens[b]:]
        ans = tok.decode(gen_ids, skip_special_tokens=True).strip()
        # keep only text after 'Final answer:' if any prompt leaked
        if "Final answer:" in ans:
            ans = ans.split("Final answer:")[-1].strip()
        answers.append(ans)
        used_ctxs.append([c["chunk_id"] for c in ctxs_list[b]])


In [ ]:
# --- Save outputs ---
# >>> Set your desired output path here <<<
OUT_PATH = "/Volumes/main/default/thesis_project/M1/Test_2.2/govern_test_M1_answers.2.2.parquet"
df_out = df.copy()
df_out["answer_M1"] = answers
df_out["retrieved_chunks"] = used_ctxs
df_out.to_parquet(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

In [ ]:
# --- Quick load & peek ---
import pandas as pd

PARQUET = "/Volumes/main/default/thesis_project/M1/Test_2.2/govern_test_M1_answers.2.2.parquet"
df = pd.read_parquet(PARQUET)

print(df.shape)              # (rows, cols)
print(df.columns.tolist())   # column names
df.head(3)                   # first few rows